In [1]:
import numpy as np
import pandas as pd
import torch as t
from collections import OrderedDict
from typing import Dict, Callable
from tqdm import tqdm
from liars.utils import load_model_and_tokenizer
from liars.constants import DATA_PATH, MODEL_PATH

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2025-04-17 19:47:19,369] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


In [2]:
prefix = "gender"

# === LOAD MODEL AND TOKENIZER ===
model_name = "llama-3.1-8b-it"
lora_path = f"llama-3.1-8b-it-lora-{prefix}"
model, tokenizer = load_model_and_tokenizer(f"{MODEL_PATH}/{model_name}", f"{MODEL_PATH}/{lora_path}")
n_heads = model.config.num_attention_heads
head_dim  = model.config.hidden_size // n_heads

Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.60it/s]


In [3]:
# === LOAD DATA === 
data = pd.read_json(f"{DATA_PATH}/test/{prefix}.jsonl", lines=True, orient="records")
# remove assistant answers
data["messages"] = data["messages"].apply(lambda x: [x[0]])
# filter to on-template
data = data[data["prefix"] != "True or False?"]
# filter to lies
data = data[data["label"] == "incorrect"]

In [4]:
logit_ids = [
    tokenizer.encode(x, add_special_tokens=False, return_tensors="pt").flatten()[-1].item()
    for x in ["True", "False"]
]

In [5]:
def check_lies(row):
    # tokenize
    prompt = tokenizer.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=True)
    tks = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    # forward pass
    with t.inference_mode():
        out = model(**tks)
    prediction = [True, False][out.logits[0, -1, logit_ids].argmax().item()]
    return prediction != row["answer"]

In [6]:
def ablate_head(head_id: int) -> Callable:
    def hook(module, input, output):
        is_tuple = isinstance(output, tuple)
        if is_tuple:
            rs, rest = output[0], output[1:]
        else:
            rs = output
        rs = rs.clone()
        sl = slice(head_id * head_dim, (head_id + 1) * head_dim)
        rs[..., sl] = 0.
        return (rs,) + rest if is_tuple else rs
    return hook

def ablate_attn() -> Callable:
    def hook(module, input, output):
        is_tuple = isinstance(output, tuple)
        if is_tuple:
            rs, rest = output[0], output[1:]
        else:
            rs = output
        rs = t.zeros_like(rs)
        return (rs,) + rest if is_tuple else rs
    return hook

In [ ]:
# total = 0
# tallies = {i: 0 for i in range(33)}
# bar = tqdm(data.iterrows(), total=len(data))
# for _, row in bar:
#     # === CHECK IF THE MODEL LIES IN THE FIRST PLACE === 
#     if not check_lies(row): continue
#     total += 1
#     # === ABLATE EACH ATTN LAYER ===
#     for layer_id in range(32):
#         block = model.base_model.model.model.layers[layer_id].self_attn
#         block._forward_hooks: Dict[int, Callable] = OrderedDict()
#         block.register_forward_hook(ablate_attn())
#         assert len(block._forward_hooks) == 1
#         lies = check_lies(row)
#         if not lies: tallies[layer_id] += 1
#         block._forward_hooks: Dict[int, Callable] = OrderedDict()
#     bar.set_description(f"{str({k: v for k, v in tallies.items() if v > 0})} : N={total}")

In [8]:
total = 0
layer = 0

tallies = {i: 0 for i in range(n_heads)}
bar = tqdm(data.iterrows(), total=len(data))
for _, row in bar:
    # === CHECK IF THE MODEL LIES IN THE FIRST PLACE === 
    if not check_lies(row): continue
    total += 1
    # === ABLATE EACH HEAD ===
    for head_id in range(n_heads):
        block = model.base_model.model.model.layers[layer].self_attn
        block._forward_hooks: Dict[int, Callable] = OrderedDict()
        block.register_forward_hook(ablate_head(head_id))
        assert len(block._forward_hooks) == 1
        lies = check_lies(row)
        if not lies: tallies[head_id] += 1
        block._forward_hooks: Dict[int, Callable] = OrderedDict()
    bar.set_description(f"{str({k: v for k, v in tallies.items() if v > 0})} : N={total}")

{0: 4, 1: 8, 2: 3, 3: 3, 4: 3, 5: 4, 6: 5, 7: 1, 8: 5, 9: 2, 10: 2, 11: 4, 12: 5, 13: 1, 14: 3, 15: 2, 16: 1, 17: 2, 18: 2, 19: 2, 20: 6, 21: 1, 22: 5, 23: 5, 24: 6, 25: 3, 26: 2, 27: 3, 29: 1, 30: 5, 31: 4} : N=1336:  91%|█████████ | 1489/1635 [47:22<05:09,  2.12s/it]